# Simulasi Instruction-Level Parallelism (ILP) & Analisis Dependensi
Mata Kuliah: **Arsitektur Komputer**  
Topik: Analisis Dependensi Data (**RAW**, **WAR**, **WAW**), Graf Dependensi (*DAG*), dan Simulasi Eksekusi Sekuensial vs Paralel (*ILP*).

In [ ]:
from dataclasses import dataclass

@dataclass
class Instruction:
    name: str
    destination: str
    sources: list

# Instruction set untuk menguji RAW, WAR, dan WAW
instructions = [
    Instruction("I1", "R1", ["R1", "R3"]),
    Instruction("I2", "R4", ["R1", "R4"]),
    Instruction("I3", "R1", ["R6", "R7"]),
    Instruction("I4", "R4", ["R8", "R9"]),
    Instruction("I5", "R6", ["R4", "R10"]),
    Instruction("I6", "R11", ["R1", "R4"]),
]

## 1. Deteksi Dependensi & Pembentukan Graph
Mendeteksi tiga jenis *data hazard*:
- **RAW (Read After Write)**: *True Data Dependency* (instruksi kedua membaca data yang dihasilkan instruksi pertama).
- **WAR (Write After Read)**: *Anti-Dependency* (instruksi kedua menulis register yang sedang dibaca instruksi pertama).
- **WAW (Write After Write)**: *Output Dependency* (kedua instruksi menulis ke register tujuan yang sama).

In [ ]:
def dependency_types(inst1, inst2):
    """Mendeteksi semua dependensi dari inst1 ke inst2."""
    dependencies = []

    # RAW: inst2 membaca register yang ditulis inst1.
    if inst1.destination in inst2.sources:
        dependencies.append(("RAW", inst1.destination))

    # WAR: inst2 menulis register yang dibaca inst1.
    if inst2.destination in inst1.sources:
        dependencies.append(("WAR", inst2.destination))

    # WAW: kedua instruksi menulis register tujuan yang sama.
    if inst1.destination == inst2.destination:
        dependencies.append(("WAW", inst1.destination))

    return dependencies


def build_dependency_graph(instructions):
    """Membentuk sisi graph sesuai urutan instruksi."""
    edges = []

    for i in range(len(instructions)):
        for j in range(i + 1, len(instructions)):
            dependencies = dependency_types(
                instructions[i], instructions[j]
            )

            if dependencies:
                edges.append((
                    instructions[i].name,
                    instructions[j].name,
                    dependencies
                ))

    return edges

## 2. Visualisasi Dependensi (Tabel & Matriks Hazard)
Fitur visualisasi untuk mempermudah analisis:
1. **Analisis Tekstual**: Daftar relasi dependensi antar instruksi.
2. **Matriks Grid Hazard (Source vs Target)**: Matriks 2 dimensi yang memperlihatkan hazard antar pasangan instruksi.
3. **Tabel Rekapitulasi Rinci**: Tabel klasifikasi tipe dependensi dan keterangan arsitekturalnya.
4. **Aliran Dependensi (DAG & Mermaid)**: Representasi graf terarah.

In [ ]:
def show_dependencies(instructions):
    """Menampilkan daftar dependensi dasar."""
    print("\n" + "=" * 60)
    print("DEPENDENCY ANALYSIS")
    print("=" * 60)

    edges = build_dependency_graph(instructions)

    for source, target, dependencies in edges:
        details = ", ".join(
            f"{kind} ({register})"
            for kind, register in dependencies
        )
        print(f"{source} -> {target}: {details}")

    return edges


def show_hazard_matrix(instructions, edges):
    """Menampilkan matriks grid dependensi antar instruksi (Source vs Target)."""
    names = [i.name for i in instructions]
    matrix = {s: {t: [] for t in names} for s in names}
    for s, t, deps in edges:
        matrix[s][t] = [f"{k}({r})" for k, r in deps]

    col_w = 18
    header = "Src \\ Tgt | " + " | ".join(f"{n:^{col_w}}" for n in names) + " |"
    div = "-" * 10 + "+" + "+".join(["-" * (col_w + 2) for _ in names]) + "+"
    print("\n" + "=" * len(header))
    print("MATRIKS HAZARD DEPENDENSI (Source -> Target)")
    print("=" * len(header))
    print(header)
    print(div)
    for s in names:
        row = f"{s:<9} | "
        for t in names:
            val = ", ".join(matrix[s][t]) if matrix[s][t] else "-"
            row += f"{val:^{col_w}} | "
        print(row)
    print(div)

    # Render HTML jika dijalankan di Jupyter Notebook
    try:
        from IPython.display import display, HTML
        headers_html = "".join(f"<th style='padding: 6px 10px; border: 1px solid #dee2e6; text-align: center; background-color: #f8f9fa;'>{n}</th>" for n in names)
        rows_html = []
        for s in names:
            cells = [f"<td style='padding: 6px 10px; border: 1px solid #dee2e6; font-weight: bold; background-color: #f8f9fa;'>{s}</td>"]
            for t in names:
                deps = matrix[s][t]
                if deps:
                    badges = []
                    for d in deps:
                        if d.startswith("RAW"):
                            bg, col = "#ffe3e3", "#c92a2a"
                        elif d.startswith("WAR"):
                            bg, col = "#fff3bf", "#d9480f"
                        else:
                            bg, col = "#d0ebff", "#1864ab"
                        badges.append(f"<span style='background-color: {bg}; color: {col}; padding: 2px 6px; border-radius: 3px; font-weight: bold; font-size: 11px;'>{d}</span>")
                    cells.append(f"<td style='padding: 6px; border: 1px solid #dee2e6; text-align: center;'>{' '.join(badges)}</td>")
                else:
                    cells.append("<td style='padding: 6px; border: 1px solid #dee2e6; text-align: center; color: #ced4da;'>-</td>")
            rows_html.append(f"<tr>{''.join(cells)}</tr>")
        display(HTML(f"<div style='margin: 15px 0; font-family: sans-serif;'><h4 style='margin-bottom: 8px;'>Matriks Grid Dependensi</h4><table style='border-collapse: collapse; font-size: 13px;'><thead><tr><th style='padding: 6px 10px; border: 1px solid #dee2e6; background-color: #f8f9fa;'>Src \\ Tgt</th>{headers_html}</tr></thead><tbody>{''.join(rows_html)}</tbody></table></div>"))
    except Exception:
        pass


def show_hazard_table(edges):
    """Menampilkan tabel rekapitulasi pasangan instruksi dan jenis hazard."""
    print("\n" + "=" * 98)
    print("TABEL REKAPITULASI DEPENDENCY HAZARDS")
    print("=" * 98)
    src_h, tgt_h, haz_h, reg_h, kat_h, ket_h = "Source", "Target", "Hazard", "Register", "Kategori", "Keterangan"
    header = f"| {src_h:<8} | {tgt_h:<8} | {haz_h:<8} | {reg_h:<8} | {kat_h:<26} | {ket_h:<30} |"
    div = "+----------+----------+----------+----------+----------------------------+--------------------------------+"
    print(div)
    print(header)
    print(div)
    desc = {
        "RAW": ("True Data Dependency", "Target membaca data hasil Source"),
        "WAR": ("Anti-Dependency", "Target menulis register yg dibaca Source"),
        "WAW": ("Output Dependency", "Keduanya menulis register tujuan yg sama")
    }
    for s, t, deps in edges:
        for k, r in deps:
            kat, ket = desc.get(k, ("-", "-"))
            print(f"| {s:<8} | {t:<8} | {k:<8} | {r:<8} | {kat:<26} | {ket:<30} |")
    print(div)


def render_dag_diagram(instructions, edges):
    """Menampilkan representasi teks DAG dan diagram Mermaid."""
    print("\n" + "=" * 65)
    print("DIRECTED ACYCLIC GRAPH (DAG) - ALIRAN DEPENDENSI")
    print("=" * 65)
    for s, t, deps in edges:
        labels = " / ".join(f"{k} ({r})" for k, r in deps)
        print(f"  {s:<3} --------> {t:<3}  : [{labels}]")

    mermaid_code = ["```mermaid", "graph TD"]
    for inst in instructions:
        mermaid_code.append(f'    {inst.name}["{inst.name}: {inst.destination} = {" + ".join(inst.sources)}"]')
    for s, t, deps in edges:
        labels = ", ".join(f"{k}({r})" for k, r in deps)
        mermaid_code.append(f"    {s} -->|{labels}| {t}")
    mermaid_code.append("```")
    mermaid_str = "\n".join(mermaid_code)

    try:
        from IPython.display import display, Markdown
        display(Markdown(mermaid_str))
    except Exception:
        pass

    return mermaid_str

## 3. Simulasi Eksekusi & Gantt Chart
- **Sequential Execution**: Menjalankan 1 instruksi per siklus (Total = jumlah instruksi).
- **ILP Parallel Execution**: Mengeksekusi secara serentak instruksi yang sudah siap (*ready* / semua dependensi pendahulunya selesai).
- **Gantt Chart**: Visualisasi pemetaan siklus eksekusi.

In [ ]:
def sequential_execution(instructions):
    print("\n" + "=" * 60)
    print("SEQUENTIAL EXECUTION")
    print("=" * 60)

    schedule = {}
    for cycle, inst in enumerate(instructions, start=1):
        schedule[cycle] = [inst.name]
        print(
            f"Cycle {cycle}: {inst.name}: "
            f"{inst.destination} = {' + '.join(inst.sources)}"
        )

    print(f"Total Sequential Cycles: {len(instructions)}")
    return len(instructions), schedule


def parallel_execution(instructions, edges):
    print("\n" + "=" * 60)
    print("INSTRUCTION-LEVEL PARALLEL EXECUTION")
    print("=" * 60)

    predecessors = {inst.name: set() for inst in instructions}

    for source, target, _ in edges:
        predecessors[target].add(source)

    completed = set()
    remaining = instructions.copy()
    cycle = 0
    schedule = {}

    while remaining:
        cycle += 1

        ready = [
            inst for inst in remaining
            if predecessors[inst.name].issubset(completed)
        ]

        if not ready:
            raise RuntimeError("Tidak ditemukan instruksi yang siap.")

        schedule[cycle] = [inst.name for inst in ready]
        print(f"\nCycle {cycle}:")

        for inst in ready:
            print(
                f"  {inst.name}: {inst.destination} = "
                f"{' + '.join(inst.sources)}"
            )

        for inst in ready:
            completed.add(inst.name)
            remaining.remove(inst)

    print(f"Total ILP Cycles: {cycle}")
    return cycle, schedule


def render_gantt_chart(title, schedule, total_cycles, instructions):
    """Menampilkan diagram timing/Gantt Chart eksekusi siklus per instruksi."""
    col_w = 7
    cycle_headers = [f"C{c}" for c in range(1, total_cycles + 1)]
    header = "Instruksi | " + " | ".join(f"{ch:^{col_w}}" for ch in cycle_headers) + " |"
    div = "-" * 10 + "+" + "+".join(["-" * (col_w + 2) for _ in range(total_cycles)]) + "+"
    print("\n" + "=" * len(header))
    print(title)
    print("=" * len(header))
    print(header)
    print(div)
    for inst in instructions:
        row = f"{inst.name:<9} | "
        for c in range(1, total_cycles + 1):
            if inst.name in schedule.get(c, []):
                row += f"{'[EX]':^{col_w}} | "
            else:
                row += f"{' . ':^{col_w}} | "
        print(row)
    print(div)

    # Render HTML jika dijalankan di Jupyter Notebook
    try:
        from IPython.display import display, HTML
        cycle_th = "".join(f"<th style='padding: 6px 12px; border: 1px solid #dee2e6; text-align: center; background-color: #f1f3f5;'>C{c}</th>" for c in range(1, total_cycles + 1))
        rows_html = []
        for inst in instructions:
            cells = [f"<td style='padding: 6px 12px; border: 1px solid #dee2e6; font-weight: bold;'>{inst.name}: {inst.destination} = {' + '.join(inst.sources)}</td>"]
            for c in range(1, total_cycles + 1):
                if inst.name in schedule.get(c, []):
                    cells.append("<td style='padding: 6px; border: 1px solid #dee2e6; text-align: center; background-color: #d3f9d8;'><span style='background-color: #2b8a3e; color: white; padding: 2px 8px; border-radius: 4px; font-weight: bold; font-size: 11px;'>EX</span></td>")
                else:
                    cells.append("<td style='padding: 6px; border: 1px solid #dee2e6; text-align: center; color: #adb5bd;'>·</td>")
            rows_html.append(f"<tr>{''.join(cells)}</tr>")
        display(HTML(f"<div style='margin: 15px 0; font-family: sans-serif;'><h4 style='margin-bottom: 8px;'>{title}</h4><table style='border-collapse: collapse; font-size: 13px;'><thead><tr><th style='padding: 6px 12px; border: 1px solid #dee2e6; background-color: #f1f3f5;'>Instruksi</th>{cycle_th}</tr></thead><tbody>{''.join(rows_html)}</tbody></table></div>"))
    except Exception:
        pass

## 4. Eksekusi Utama (Main Program)
Menjalankan seluruh alur deteksi hazard, simulasi eksekusi, perbandingan performa (*Speedup* & *Efficiency*), dan visualisasi.

In [ ]:
def main():
    print("=" * 60)
    print("INSTRUCTION-LEVEL PARALLELISM SIMULATOR")
    print("=" * 60)

    print("\nInstruction Set:")
    for inst in instructions:
        print(
            f"{inst.name}: {inst.destination} = "
            f"{' + '.join(inst.sources)}"
        )

    # 1. Analisis & Visualisasi Dependensi
    edges = show_dependencies(instructions)
    show_hazard_matrix(instructions, edges)
    show_hazard_table(edges)

    # 2. Simulasi Eksekusi
    sequential_cycles, seq_schedule = sequential_execution(instructions)
    parallel_cycles, par_schedule = parallel_execution(instructions, edges)

    # 3. Visualisasi Gantt Chart Eksekusi
    render_gantt_chart(
        f"GANTT CHART: SEQUENTIAL EXECUTION ({sequential_cycles} Cycles)",
        seq_schedule,
        sequential_cycles,
        instructions
    )
    render_gantt_chart(
        f"GANTT CHART: ILP PARALLEL EXECUTION ({parallel_cycles} Cycles)",
        par_schedule,
        parallel_cycles,
        instructions
    )

    # 4. Perbandingan Performa
    speedup = sequential_cycles / parallel_cycles
    efficiency = speedup / len(instructions) * 100

    print("\n" + "=" * 60)
    print("PERFORMANCE COMPARISON")
    print("=" * 60)
    print(f"Sequential Cycles : {sequential_cycles}")
    print(f"ILP Cycles        : {parallel_cycles}")
    print(f"Speedup           : {speedup:.2f}x")
    print(f"ILP Efficiency    : {efficiency:.2f}%")

    # 5. Visualisasi Aliran Graf Dependensi
    render_dag_diagram(instructions, edges)


if __name__ == "__main__":
    main()